## DATA 622 - Assignment 2: 

### Introduction
Compare the results with the results from previous homework.

Read the following articles:
- https://www.hindawi.com/journals/complexity/2021/5550344/
- https://www.ncbi.nlm.nih.gov/pmc/articles/PMC8137961/

Search for academic content (at least 3 articles) that compare the use of decision trees vs SVMs in your current area of expertise.

Perform an analysis of the dataset used in Homework #2 using the SVM algorithm. Compare the results with the results from previous homework. Answer questions, such as:
- Which algorithm is recommended to get more accurate results?
- Is it better for classification or regression scenarios?
- Do you agree with the recommendations?
- Why?

### Literature Review

There are numerous studies that compares the predictive performance between Decision Tree models and Support Vector Machine model in various fields, particularly in the financial industry. In fact, there are many studies that often focusing on forecasting. Per Fan J. (2023) paper, the author conducted a comparative analysis of SVM and Decision Trees on credit default predictions. For the dataset, the author used the UCI Credit Card Fraud dataset and later evaluated and compared the performance of each model with standard metrics including Mean Absolute Error, R-Squared, and Mean Squared Error. The result showed that the Decision tree model is better in forecasting credit default rate than SVM. The SVM scored lower than the Decision Trees in Mean Absolute Error and Mean Squared Error.

Per Golbayani, P., Florescu, I., & Chatterjee, R. (2020) paper, the team examined various machine learning models in corporate credit rating forecasting. They chose Bagged Decision Trees, Random Forest, support vector machine and Multilayer Perceptron and utilized a dataset that consists of stocks from various fields and industries. From there, the team applied a 10-fold cross validation procedure to ensure better accuracy and overfitting reduction and used a new measure called Notch measure to evaluate the performance. The Notch Measure is the distance between predictions and real ratings as Notches distance. The results showed that Bagged Decision Tree and Random Forests performed better than ANN and SVM. 

Per Wang, J. (2025) paper, the author explored the machine learning technique in stock price forecasting. The author examined and evaluated three different models: Support Vector Machines (SVM), Decision Trees, and Random Forests. For the dataset, the author chose to use the historical data of Tesla from 2010 to 2023 and included 20-day Simple Moving Average as another key indicator. To evaluate each model’s performance, the author utilized accuracy, precision, recall, and F1 metrics. As a result, the SVM had the best performance in accuracy, recall, and F1 score compared to Decision Trees and Random Forests. The author interestingly found that while SVM scored better, the Random Forest was more consistent with predictions and had fewer errors.


In [18]:
## load modules
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import datetime as dt
import seaborn as sns
import plotly.express as px
import matplotlib.dates as mdates

## load svm modules
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix
)
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report

## Load Data

In [19]:
bank_full_url = 'https://raw.githubusercontent.com/eddiexunyc/ml_big_data_work/refs/heads/main/Assignment%201/bank/bank-full.csv'
bank_full = pd.read_csv(bank_full_url, sep=';')
bank_full.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45211 entries, 0 to 45210
Data columns (total 17 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   age        45211 non-null  int64 
 1   job        45211 non-null  object
 2   marital    45211 non-null  object
 3   education  45211 non-null  object
 4   default    45211 non-null  object
 5   balance    45211 non-null  int64 
 6   housing    45211 non-null  object
 7   loan       45211 non-null  object
 8   contact    45211 non-null  object
 9   day        45211 non-null  int64 
 10  month      45211 non-null  object
 11  duration   45211 non-null  int64 
 12  campaign   45211 non-null  int64 
 13  pdays      45211 non-null  int64 
 14  previous   45211 non-null  int64 
 15  poutcome   45211 non-null  object
 16  y          45211 non-null  object
dtypes: int64(7), object(10)
memory usage: 5.9+ MB


### Pre-processing

In [20]:
# convert the target variable to binary
bank_full['y'] = bank_full['y'].map({'yes': 1, 'no': 0})

# encode the categorical variables
cat_cols = bank_full.select_dtypes(include='object').columns
bank_encoded = pd.get_dummies(bank_full, columns=cat_cols, drop_first=True)

print("Encoded dataset shape:", bank_encoded.shape)
bank_encoded

Encoded dataset shape: (45211, 43)


,age,balance,day,duration,campaign,pdays,previous,y,job_blue-collar,job_entrepreneur,...,month_jul,month_jun,month_mar,month_may,month_nov,month_oct,month_sep,poutcome_other,poutcome_success,poutcome_unknown
0,58,2143,5,261,1,-1,0,0,0,0,...,0,0,0,1,0,0,0,0,0,1
1,44,29,5,151,1,-1,0,0,0,0,...,0,0,0,1,0,0,0,0,0,1
2,33,2,5,76,1,-1,0,0,0,1,...,0,0,0,1,0,0,0,0,0,1
3,47,1506,5,92,1,-1,0,0,1,0,...,0,0,0,1,0,0,0,0,0,1
4,33,1,5,198,1,-1,0,0,0,0,...,0,0,0,1,0,0,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45206,51,825,17,977,3,-1,0,1,0,0,...,0,0,0,0,1,0,0,0,0,1
45207,71,1729,17,456,2,-1,0,1,0,0,...,0,0,0,0,1,0,0,0,0,1
45208,72,5715,17,1127,5,184,3,1,0,0,...,0,0,0,0,1,0,0,0,1,0
45209,57,668,17,508,4,-1,0,0,1,0,...,0,0,0,0,1,0,0,0,0,1


In [21]:
# drop the target variable
bank_feature = bank_encoded.drop('y', axis=1)
bank_y = bank_encoded['y']

# split the data into train and test subset
X_train, X_test, y_train, y_test = train_test_split(bank_feature, bank_y, test_size=0.2, random_state=42, stratify=bank_y)
print("Train size:", X_train.shape, "| Test size:", X_test.shape)

Train size: (36168, 42) | Test size: (9043, 42)


### Support Vector Machine Model

In [22]:
# define the scale for the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [24]:
# train the svm classifier
svm_clf = SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42, probability=True)
svm_clf.fit(X_train_scaled, y_train)

# evaluate the model
y_pred = svm_clf.predict(X_test_scaled)
y_prob = svm_clf.predict_proba(X_test_scaled)[:, 1] 

# ROC-AUC
auc = roc_auc_score(y_test, y_prob)

print("\nROC-AUC Score:", auc)
print("SVM Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))


ROC-AUC Score: 0.905292354639429
SVM Accuracy: 0.9025765785690589

Classification Report:
               precision    recall  f1-score   support

           0       0.92      0.98      0.95      7985
           1       0.68      0.32      0.43      1058

    accuracy                           0.90      9043
   macro avg       0.80      0.65      0.69      9043
weighted avg       0.89      0.90      0.89      9043


Confusion Matrix:
 [[7824  161]
 [ 720  338]]


### Insight and Observation

The Support Vector Machine is examined and evaluated by using the same dataset from prior assignments and undergoes the same pre-processing that was applied to the Decision Tree models.

### References

- Fan, J. (2023). Predicting of Credit Default by SVM and Decision Tree Model Based on Credit Card Data. BCP Business & Management, 38, 28–33. https://doi.org/10.54691/bcpbm.v38i.3666
‌

- Golbayani, P., Florescu, I., & Chatterjee, R. (2020). A comparative study of forecasting corporate credit ratings using neural networks, support vector machines, and decision trees. The North American Journal of Economics and Finance, 54, 101251. https://doi.org/10.1016/j.najef.2020.101251
‌
- Wang, J. (2025). Tesla stock prediction with SVM, decision tree and random forest. Highlights in Business, Economics and Management, 50, 342–346. https://doi.org/10.54097/6xrr2902
‌
